# Projet VRE
## ARGAUD Myriam, SAAD Maria

## Supervised problem definition

The objective of this notebook is to predict the onshore winf production by predicting the cp_factor base on meteorological and energy demand data.

Our idea is to do this region by region, by defining for each region a couple (x,y).x is the input variables (wind,temperature, energy demand, geopotential height, surface density). y is the variable we want to predict, the capacity factor.

To attain this objective, we use machine learning methods. The first part consists of an investigation by hand and generating first linear models. The second part consists in building a linear model by using built-in python functions for machine learning. The last part consists in investigating how to improve our predictions, in particular
non-linear models to improve the prediction.

# I. Investigation by hand

In [ ]:
#Import the modules
from pathlib import Path
from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
import numpy as np
import matplotlib.colors as mcolors
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.decomposition import PCA

## Import the data

The first cell is to import the data from Google Drive, because we worked on Google Collab to be able to work on the code simultaneously.

In [ ]:
!pip install netCDF4

In [ ]:
#Open and read the data, adapted from the notebook provided by the professor

# Directories the data is saved
data_dir_energy = Path('../data/energy_france')
data_dir_climate = Path('../data/climate_france_2019')

# Template filenames
filename_mask = 'mask_datagouv_french_regions_merra2_Nx_France.nc'
filename_climate = 'merra2_area_selection_output_{}_merra2_2019-2019.nc'
filename_energy = 'reseaux_energies_{}.csv'

#Open the files and store the data in dataframes
df_meridional = xr.open_dataset('../data/climate_france_2019/merra2_area_selection_output_upper_meridional_wind_merra2_2019-2019.nc', engine='netcdf4')
df_zonal = xr.open_dataset('../data/climate_france_2019/merra2_area_selection_output_upper_zonal_wind_merra2_2019-2019.nc', engine='netcdf4')

df_temperature = xr.open_dataset('../data/climate_france_2019/merra2_area_selection_output_surface_temperature_merra2_2019-2019.nc', engine='netcdf4')
df_humidity = xr.open_dataset('../data/climate_france_2019/merra2_area_selection_output_surface_specific_humidity_merra2_2019-2019.nc', engine='netcdf4')

df_surface_density = xr.open_dataset('../data/climate_france_2019/merra2_area_selection_output_surface_density_merra2_2019-2019.nc', engine='netcdf4')
df_height = xr.open_dataset('../data/climate_france_2019/merra2_area_selection_output_height_500_merra2_2019-2019.nc', engine='netcdf4')

df_demand = pd.read_csv('../data/energy_france/reseaux_energies_demand_demand.csv')
df_cp_factor = pd.read_csv('../data/energy_france/reseaux_energies_capacityfactor_wind-onshore.csv')


FileNotFoundError: [Errno 2] No such file or directory: '/data/climate_france_2019/merra2_area_selection_output_upper_meridional_wind_merra2_2019-2019.nc'

In [ ]:
# Read and plot grid point-region mask
ds_mask = xr.open_dataset('../data/climate_france_2019/mask_datagouv_french_regions_merra2_Nx_France.nc', engine='netcdf4')
da_mask = ds_mask['mask']

colors = plt.cm.tab20(np.linspace(0, 1, 14))  # par ex. jusqu’à 20 couleurs
cmap = mcolors.ListedColormap(colors)

plt.figure()
sc = plt.scatter(da_mask['lon'], da_mask['lat'], c=da_mask, cmap=cmap)

handles, labels = sc.legend_elements(num=None)
plt.legend(handles, labels, title="Code région")
plt.show()

da_mask_dataframe = da_mask.to_dataframe()
print(da_mask_dataframe)

liste_regions = [(3,'Hauts-de-France'),(11,'Normandie'),(8,'Île-de-France'),(2,'Grand Est'),(6,'Bretagne'),(12,'Pays de la Loire'),(7,'Centre-Val de Loire'),(5,'Bourgogne-Franche-Comté'),(3,'Nouvelle-Aquitaine'),(4,'Auvergne-Rhône-Alpes'),(9,'Occitanie'),(13,"Provence-Alpes-Côte d'Azur")]


## Treat the data

In [ ]:
#Create the dataframe containing the data

ds = df_meridional.copy()
ds["upper_zonal_wind"]         = df_zonal["upper_zonal_wind"]
ds["surface_temperature"]      = df_temperature["surface_temperature"]
ds["surface_specific_humidity"] = df_humidity["surface_specific_humidity"]
ds["surface_density"]          = df_surface_density["surface_density"]
ds["height_500"]               = df_height["height_500"]

#Convert into a dataframe
df_new = ds.to_dataframe().reset_index()

#Rename the columns
df_new = df_new.rename(columns={
    "upper_meridional_wind": "meridional_wind",
    "upper_zonal_wind": "zonal_wind",
    "surface_temperature": "temperature",
    "surface_specific_humidity": "humidity",
    "surface_density": "surface_density",
    "height_500": "height",
})


In [ ]:
#We apply the regional mask to the data, to separate it into regions

#Convert the mask into dataframe so it is easier to handle
df_mask = da_mask.to_dataframe().reset_index()
df_mask = df_mask.rename(columns={'mask': 'region'})

#Apply the mask to the data
df_new['lat_round'] = df_new['lat'].round(4)
df_new['lon_round'] = df_new['lon'].round(4)
df_mask['lat_round'] = df_mask['lat'].round(4)
df_mask['lon_round'] = df_mask['lon'].round(4)

df_merged = df_new.merge(df_mask[['lat_round', 'lon_round', 'region']],
                         on=['lat_round', 'lon_round'],
                         how='left')

#Clean
df_merged = df_merged.drop(columns=['lat_round', 'lon_round'])

#Create a dictionnary to
regions_dict = {}
unique_regions = df_merged['region'].dropna().unique()

for region_id in sorted(unique_regions):
    region_df = df_merged[df_merged['region'] == region_id].copy()
    regions_dict[f'region_{int(region_id)}'] = region_df
    print(f"Région {int(region_id)}: {len(region_df)} points")

print(df_merged.head(50))


# Groupby sur time ET region, moyenne sur lat/lon
df_spatial_mean = df_merged.groupby(['time', 'region']).mean().reset_index()

print("Visualize the data")
print(df_spatial_mean.head(20))

## Create a dataframe for each region
We are forced to use meteorological data that is averaged by region and by month, because the capacity factor that we have is only one value per region per month.

In [ ]:
regions_timeseries = {}

for region_id in sorted(df_spatial_mean['region'].unique()):
    #Extract the data for the region identified with region_id
    df_region = df_spatial_mean[df_spatial_mean['region'] == region_id].copy()

    #Sort by time
    df_region = df_region.sort_values('time').reset_index(drop=True)

    #Store in a dictionnary
    regions_timeseries[f'region_{int(region_id)}'] = df_region

    print(regions_timeseries['region_2'])

#Add the demand data

for (region_id,region) in liste_regions:
    regions_timeseries[f'region_{int(region_id)}']['demand'] = df_demand[f'{region}']

#print(regions_timeseries['region_2'])

#Add the data of capacity factor
for (region_id,region) in liste_regions:
    regions_timeseries[f'region_{int(region_id)}']['cp_factor'] = df_cp_factor[f'{region}']

#Clean the data from stacked_dim, lat,lon
for (region_id,region) in liste_regions:
    regions_timeseries[f'region_{int(region_id)}'].drop(columns = ['stacked_dim', 'lat', 'lon','time'])

#print(regions_timeseries['region_2'])


## Create correlation matrices
For each region, we create a correlation matrix, just to see the relation between the variables.

In [ ]:
correlation_matrices = {}

for (region_id,region) in liste_regions:
    corr_mat = regions_timeseries[f'region_{int(region_id)}'].corr(method='pearson')    #We use the classic OLS
    correlation_matrices[f'region_{int(region_id)}'] = corr_mat

corr_mat = regions_timeseries['region_2'].corr(method='pearson')
print(corr_mat)

Then we want to investigate, for each region, which variables are correlated with a correlation factor over 0.7.

In [ ]:
#Define the threshold above which we consider high correlation between variables.
seuil = 0.7
correlated_pairs = {}

for (region_id,region) in liste_regions:
    key = f'region_{int(region_id)}'
    df_reg = regions_timeseries[key]

    # matrice de corrélation de la région
    corr_mat = df_reg.corr(method='pearson')

    pairs = []
    cols = corr_mat.columns

    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            var1 = cols[i]
            var2 = cols[j]
            r = corr_mat.iloc[i, j]
            if abs(r) > seuil:
                pairs.append((var1, var2, r))

    correlated_pairs[key] = pairs


#Print for one region to visualize the results
for key, pairs in correlated_pairs.items():
    print(f"\n{key}:")
    for var1, var2, r in pairs:
        print(f"  {var1} - {var2}: r = {r:.3f}")



We notice that the correlation values are different according to the region. We now have to choose which variables to drop for each region.

    region3,11,8,2,6,12,7,5 : We can drop surface_density as it is higly correlated to humidity.
    region13 : we can drop also surface_density, and height as they are highly correlated to temperature.
    region9,4 : we can also drop surface_density as it is highly correlated to temperature and humidity.


In [ ]:
for index in [3,11,8,2,6,12,7,5,9,4] :
    regions_timeseries[f'region_{int(index)}'].drop(columns = ['surface_density'])

for index in [13] :
    regions_timeseries[f'region_{int(index)}'].drop(columns = ['surface_density','height'])



## Generate models for each region
In the first time, we generate models using basic OLS.

In [ ]:
#The input matrices are the dataframes from regions_timeseries without the cp. The output matrices are the cp columns.
INPUT_matrices = {}
OUTPUT_matrices = {}

for (region_id,region) in liste_regions:
    key = f'region_{int(region_id)}'

    input_matrix = regions_timeseries[key].drop(columns='cp_factor')
    output_matrix = regions_timeseries[key]['cp_factor']

    INPUT_matrices[key] = input_matrix
    OUTPUT_matrices[key] = output_matrix

print(INPUT_matrices['region_2'])
print('OUTPUT',OUTPUT_matrices['region_2'])

In [ ]:
def regression(key):

    #This function generates a model based on the input matrix we created (demand and meteorologic data) and the output matrix we just created (cp factor).
    # input : key. This is the numerical index refering to the region.

    X = INPUT_matrices[key]
    y = OUTPUT_matrices[key]

    #Crop the column time, otherwise the regression would not work
    if 'time' in X.columns:
        X = X.drop(columns=['time'])

    #Keep only numeric columns
    X = X.select_dtypes(include=['number'])

    #Crop every line with a NaN
    df_xy = pd.concat([X, y], axis=1).dropna()
    X_clean = df_xy[X.columns]
    y_clean = df_xy[y.name]

    INPUT_matrices[key] = X_clean
    OUTPUT_matrices[key] = y_clean

    #Split in train / test sets
    X_train, X_test, y_train, y_test = train_test_split(X_clean, y_clean, test_size=0.2, random_state=0)

    #Compute the linear Model
    model = LinearRegression()
    model.fit(X_train, y_train)

    #Compute the error
    y_pred = model.predict(X_test)
    R2 = r2_score(y_test, y_pred)

    return model, R2


In [ ]:
models = {}
scores = {}

for key in INPUT_matrices.keys():
    model, R2 = regression(key)
    scores[key] = R2
    models[key] = model

print(scores)  # R² par région

# II. Using built-in python functions for machine learning
Now that we understand the data sets and the situation better, we can generate models using built-in python functions for machine learning.

In [ ]:
# We define a dataset used for the analysis and the models. Each row corresponds to one month and one region.
# The target variable is the wind capacity factor. The other columns are climate features for each region on an average of a month
region_id_to_name = dict(zip(ds_mask["region_index"].values, ds_mask["region"].values)) #building a mapping

#we make a copy of the climate features
df_climate = df_spatial_mean.copy()
df_climate["region"] = df_climate["region"].astype(int)
df_climate["region"] = df_climate["region"].map(region_id_to_name)

df_climate = df_climate.drop(columns=["region"]) #keep only the usefull columns

df_X = (  #climate fetaures
    df_climate
    .set_index(["time", "region"])
    .sort_index()
)


df_y = (   #capacity factor
    df_cp_factor.rename_axis("time")
    .stack()
    .rename("capacity_factor")
    .rename_axis(index=["time", "region"])
    .to_frame()
    .sort_index()
)
df_data = df_y.join(df_X, how="inner").dropna()  #join the target feature to the climate features

display(df_data.head())

In [ ]:
# We are using the spearman method to do the correlation analysis

regions = df_data.index.get_level_values("region").unique().tolist()
print("Number of regions:", len(regions))
print("Regions:", regions)

def corr_with_cf(df, region, method="spearman"):
    df_r = df.xs(region, level="region")
    corr = df_r.corr(method=method)["capacity_factor"].drop("capacity_factor")
    corr = corr.reindex(corr.abs().sort_values(ascending=False).index)
    return corr



In [ ]:
# We plot the correlations for the region BRetagne

region = "Bretagne"
corr_bretagne = corr_with_cf(df_data, region, method="spearman")

print("\nTop correlations for", region)
display(corr_bretagne.head(12))

top_n = 12
plt.figure(figsize=(8, 4))
corr_bretagne.head(top_n).sort_values().plot(kind="barh")
plt.title(f"Top {top_n} Spearman correlations with capacity factor ({region})")
plt.xlabel("Spearman correlation")
plt.tight_layout()
plt.show()


In [ ]:
# We do it for all the regions
# we select the top 5 of the features the most correlated with the capacity factor

k = 5
top_corr_per_region = {}

for r in regions:
    corr_r = corr_with_cf(df_data, r, method="spearman")
    top_corr_per_region[r] = corr_r.head(k)

top_corr_table = pd.DataFrame(top_corr_per_region).T
print("\nTop", k, "correlated features per region (Spearman):")
display(top_corr_table)

# the features that appear the most in top
all_top_features = []
for r in regions:
    all_top_features += top_corr_per_region[r].index.tolist()

freq = pd.Series(all_top_features).value_counts()
print("\nMost frequent features in the top", k, "across regions:")
display(freq.head(15))

In [ ]:
# now if we take the region of bretagne for instance we see what features are correlated between eachother in order to reduce the dimension
# we will reduce the dimension byy using the method of PCA
region = "Bretagne"
df_r = df_data.xs(region, level="region")

X = df_r.drop(columns=["capacity_factor"])

corr_features = X.corr(method="pearson")

plt.figure(figsize=(8, 6))
plt.imshow(corr_features, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.title(f"Feature correlation matrix ({region})")
plt.xticks(range(len(X.columns)), X.columns, rotation=90)
plt.yticks(range(len(X.columns)), X.columns)
plt.tight_layout()
plt.show()


In [ ]:
def safe_time_splits(n_samples, wanted_splits=5):
    """
    the problem here is that we don't have that much of dataset because we did the mean over a month (because we have the capacity factor monthly only during the year 2019.
    so if we create too many splits (as we did before and the result wasn't satifying, it can create tiny training folds and break PCA.
    """
    if n_samples <= 8:
        return 2
    if n_samples <= 12:
        return min(3, wanted_splits)
    return wanted_splits


def rf_topk_features(X, y, k=5, random_state=0):
    rf = RandomForestRegressor(
        n_estimators=500,
        max_depth=4,          #we take it small to  avoid overfitting
        random_state=random_state
    )
    rf.fit(X, y)
    imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
    return imp.index[:k].tolist(), imp


def score_model_cv(model, X, y, n_splits):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores = cross_val_score(model, X, y, cv=tscv, scoring="r2")
    return float(np.nanmean(scores)), scores

# This function trains and evaluates simple models for one region. It Selects the data for one region (monthly values), Separate target (capacity_factor) and climate features. Then, we Use a Random Forest to rank feature importance and keep the k most important variables.
#    A simple linear model, the Ridge regression, is then trained using only these selected features.
# We then Apply PCA to the full feature set to reduce the number of variables by projection. (to have a better model)
# A linear regression model is trained on the PCA components.
# Evaluate all models using a time series cross-validation (train on past months,test on future months).
# Return the model scores and selected variables
# The goal is to compare feature selection (Random Forest) and dimensionality reduction (PCA) for wind capacity factor modeling, region by region.

def train_region_models(df_data, region, rf_k=5, pca_components=3, wanted_splits=5, random_state=0):
    df_r = df_data.xs(region, level="region").copy()
    y = df_r["capacity_factor"]
    X = df_r.drop(columns=["capacity_factor"])

    # Choose a safe number of splits for small samples
    n_splits = safe_time_splits(len(X), wanted_splits=wanted_splits)

    # linear model on selected features
    top_vars, rf_importance = rf_topk_features(X, y, k=min(rf_k, X.shape[1]), random_state=random_state)
    X_rf = X[top_vars]

    #Use a ridge regression
    rf_selected_model = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1.0))
    ])

    r2_rfsel_mean, r2_rfsel_folds = score_model_cv(rf_selected_model, X_rf, y, n_splits)

    # PCa Linear Regression on principal components
    # to not have an error PCA components must be <= min(n_features, min_train_samples)
    min_train = len(X) // (n_splits + 1)
    ncomp_safe = min(pca_components, X.shape[1], max(1, min_train))

    pca_model = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=ncomp_safe)),
        ("lr", LinearRegression())
    ])

    r2_pca_mean, r2_pca_folds = score_model_cv(pca_model, X, y, n_splits)

    baseline_cols = [c for c in X.columns if c == "wind_speed_mean"]
    if len(baseline_cols) == 1:
        X_base = X[baseline_cols]
        base_model = Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LinearRegression())
        ])
        r2_base_mean, r2_base_folds = score_model_cv(base_model, X_base, y, n_splits)
    else:
        r2_base_mean, r2_base_folds = np.nan, np.array([])

    return {
        "region": region,
        "n_samples": len(X),
        "n_features": X.shape[1],
        "n_splits": n_splits,

        "baseline_r2_mean": r2_base_mean,
        "rfsel_r2_mean": r2_rfsel_mean,
        "pca_r2_mean": r2_pca_mean,

        "rf_selected_vars": top_vars,
        "pca_n_components": ncomp_safe,

        "rf_importance": rf_importance,
        "rfsel_r2_folds": r2_rfsel_folds,
        "pca_r2_folds": r2_pca_folds,
        "baseline_r2_folds": r2_base_folds
    }


#we do this for all the regions

regions = df_data.index.get_level_values("region").unique().tolist()

all_results = []
details = {}

for r in regions:
    res = train_region_models(
        df_data,
        r,
        rf_k=5,
        pca_components=3,
        wanted_splits=5,
        random_state=0
    )
    all_results.append({
        "region": res["region"],
        "n_samples": res["n_samples"],
        "n_features": res["n_features"],
        "n_splits": res["n_splits"],
        "baseline_r2_mean": res["baseline_r2_mean"],
        "rfsel_r2_mean": res["rfsel_r2_mean"],
        "pca_r2_mean": res["pca_r2_mean"],
        "pca_n_components": res["pca_n_components"]
    })
    details[r] = res

results_df = pd.DataFrame(all_results).set_index("region").sort_values("rfsel_r2_mean", ascending=False)
results_df



In [ ]:
#we plot the comparasion for all the regions

results_df[["baseline_r2_mean", "rfsel_r2_mean", "pca_r2_mean"]].plot(kind="bar", figsize=(12,4))
plt.title("Regional model performance (R2) — baseline vs RF selection vs PCA")
plt.ylabel("Mean CV R2")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


# III. Improving the predictions
Seeing previous results for R², we want to improve the models to have a more accurate preivision.

## Learning curve

In [ ]:
liste_regions = [(3,'Hauts-de-France'),(11,'Normandie'),(8,'Île-de-France'),(2,'Grand Est'),(6,'Bretagne'),(12,'Pays de la Loire'),(7,'Centre-Val de Loire'),(5,'Bourgogne-Franche-Comté'),(3,'Nouvelle-Aquitaine'),(4,'Auvergne-Rhône-Alpes'),(9,'Occitanie'),(13,"Provence-Alpes-Côte d'Azur")]


In [ ]:
region = 'Bretagne'
key = 'region_6'
X = INPUT_matrices[key]
y = OUTPUT_matrices[key]

estimator = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])

train_sizes, train_scores, test_scores = learning_curve(
    estimator=estimator,
    X=X,
    y=y,
    train_sizes=np.linspace(0.2, 1.0, 5),  # 20%, 40%, 60%, 80%, 100%
    cv=5,                                   # 5-fold CV (attention : mélange le temps)
    scoring="r2",
    shuffle=True,
    random_state=0
)

train_scores_mean = train_scores.mean(axis=1)
test_scores_mean = test_scores.mean(axis=1)

plt.figure(figsize=(6,4))
plt.plot(train_sizes, train_scores_mean, "o-", label="R2 train")
plt.plot(train_sizes, test_scores_mean, "o-", label="R2 validation")
plt.xlabel("Nombre d'échantillons d'entraînement")
plt.ylabel("R2 moyen (CV)")
plt.title("Courbe d'apprentissage (Ridge)")
plt.legend()
plt.tight_layout()
plt.show()


We want to compare the results obtained with the different models.

In [ ]:
def build_Xy_region(key):
    #This function builds the X and y matrices for the region
    df_reg = regions_timeseries[key].copy()

    X = df_reg.drop(columns=['cp_factor', 'time'])
    y = df_reg['cp_factor']

    X = X.select_dtypes(include=['number'])
    df_xy = pd.concat([X, y], axis=1).dropna()
    X, y = df_xy[X.columns], df_xy[y.name]

    n = len(X)
    split = int(0.8 * n)
    X_train, X_test = X.iloc[:split], X.iloc[split:]
    y_train, y_test = y.iloc[:split], y.iloc[split:]
    return X_train, X_test, y_train, y_test


In [ ]:
def compare_models(key):
    X_train, X_test, y_train, y_test = build_Xy_region(key, with_time_feats=True)

    # Simple linear regression (OLS)
    ols = LinearRegression()
    ols.fit(X_train, y_train)
    R2_ols = r2_score(y_test, ols.predict(X_test))

    #Ridge regression
    ridge = Ridge(alpha=10.0)
    ridge.fit(X_train, y_train)
    R2_ridge = r2_score(y_test, ridge.predict(X_test))

    #Random forest
    rf = RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=5,
        random_state=0,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    R2_rf = r2_score(y_test, rf.predict(X_test))

    return R2_ols, R2_ridge, R2_rf, rf

# Exemple pour region_2
key = 'region_2'
R2_ols, R2_ridge, R2_rf, rf_model = compare_models(key)

print(f"{key} – R² test OLS   : {R2_ols:.3f}")
print(f"{key} – R² test Ridge : {R2_ridge:.3f}")
print(f"{key} – R² test RF    : {R2_rf:.3f}")

